# Unfallatlas Deutschland — A³-Phase

**Phase:** Algorithm / Adapt / Adjust (A³) · 3 of 5 · QUA³CK
**Goal of this notebook:** implement the U-phase §10 preprocessing contract,
train and compare baseline and tree-ensemble models, select and tune the
best imbalance-aware configuration, and report exactly one held-out
test-2024 evaluation against the Q-phase acceptance criteria.

**Strict scope.** This notebook does not compute SHAP values, does not
discuss literature or limitations, and does not touch the Streamlit app —
those are Phase C and Phase K. See
`docs/superpowers/plans/2026-07-01-a3-phase-modelling.md` for the full
scope boundary.

---

## Position in the QUA³CK process

| Phase | Notebook | Status |
|:---|:---|:---:|
| Q — Question | `01_Q_Phase.ipynb` | ✓ |
| U — Understanding | `02_U_Phase.ipynb` | ✓ |
| **A³ — Algorithm / Adapt / Adjust** | `03_A3_Phase.ipynb` | **→ here** |
| C — Conclude & Compare | `04_C_Phase.ipynb` | pending |
| K — Knowledge Transfer | `app/streamlit_app.py` | pending |

---

## 0 — Setup and reproducibility

In [ ]:
import json
import subprocess
import time
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import optuna
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, cross_val_score

from unfallatlas.features.preprocessing import (
    build_preprocessor,
    chronological_split,
    load_training_frame,
    split_features_target,
)
from unfallatlas.models.baseline import (
    build_logreg_pipeline,
    build_majority_class_classifier,
    build_random_guess_classifier,
)
from unfallatlas.models.boosting import (
    build_catboost_pipeline,
    build_lightgbm_pipeline,
    build_random_forest_pipeline,
    build_xgboost_pipeline,
    gpu_available,
)
from unfallatlas.models.evaluate import evaluate_predictions, meets_acceptance_criteria
from unfallatlas.models.imbalance import (
    balanced_sample_weight,
    find_best_threshold_for_class,
    resample_adasyn,
    resample_smote,
)
from unfallatlas.models.ordinal import build_ordinal_pipeline

In [ ]:
pio.templates.default = "plotly_white"
pio.renderers.default = "vscode"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
np.random.seed(42)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG_DIR = BASE_DIR / "reports" / "figures" / "a3_phase"
FIG_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def _git_short_sha():
    try:
        return (
            subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
            )
            .decode()
            .strip()
        )
    except Exception:
        return "unknown"

### GPU acceleration (optional, machine-specific — not part of the reproducible contract)

`USE_GPU` controls XGBoost/LightGBM/CatBoost's training device:
- `None` (default) — auto-detect via `nvidia-smi`; falls back to CPU with
  no code change on a machine without a CUDA GPU (e.g. a grader's).
- `True` — force GPU (fails loudly if no compatible GPU/driver is present).
- `False` — force CPU everywhere, even if a GPU is available.

Random Forest and Logistic Regression always run on CPU (scikit-learn has
no GPU backend without a separate RAPIDS/cuML environment, which this
project does not depend on).

In [ ]:
USE_GPU = None  # None = auto-detect, True = force GPU, False = force CPU

_use_gpu_resolved = gpu_available() if USE_GPU is None else USE_GPU
print(f"GPU acceleration: {'ON' if _use_gpu_resolved else 'OFF'}  (USE_GPU={USE_GPU})")

### Progress logging and per-model checkpointing

`nbconvert --execute` does not stream cell output live and only writes the
`.ipynb` file once the whole run finishes (or crashes) — so without an
explicit external log, there is no way to see progress while a long cell is
running, and no way to recover already-fitted models if a later cell fails.

`_log_progress()` appends timestamped lines directly to
`reports/a3_progress.log` (flushed immediately, so `tail -f` shows it live).
`_fit_or_checkpoint()` saves each fitted pipeline to
`data/processed/a3_checkpoints/<git-commit>/` right after training and
reloads from there on a re-run instead of refitting — so a crash in, say,
CatBoost training does not require redoing Random Forest/XGBoost/LightGBM.

The checkpoint directory is scoped by the current short git commit hash
(uncommitted changes still share the last commit's directory — clear
`data/processed/a3_checkpoints/` manually while actively iterating on
uncommitted model-builder changes). Any *committed* change to a model
builder's hyperparameters lands in a new commit, which gets a fresh,
empty checkpoint directory automatically — a stale checkpoint from a
previous configuration can never silently be loaded as if it reflected
the current code.

In [ ]:
PROGRESS_LOG = BASE_DIR / "reports" / "a3_progress.log"
PROGRESS_LOG.parent.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROCESSED_DIR / "a3_checkpoints" / _git_short_sha()
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def _log_progress(message: str) -> None:
    timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"
    line = f"[{timestamp}] {message}"
    print(line)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")
        f.flush()


def _fit_or_checkpoint(name: str, build_fn, fit_kwargs: dict | None = None):
    """Load a cached fitted pipeline for `name` if present, else fit + save it.

    Checkpoints live under CHECKPOINT_DIR, which is scoped by the current
    git commit hash — so a committed change to any model builder's
    hyperparameters automatically gets a fresh cache directory instead of
    silently reusing a stale checkpoint from a different configuration.
    """
    checkpoint_path = CHECKPOINT_DIR / f"{name}.joblib"
    if checkpoint_path.exists():
        _log_progress(f"  -> {name}: loaded from checkpoint ({checkpoint_path.name})")
        return joblib.load(checkpoint_path)
    pipeline = build_fn()
    if fit_kwargs:
        pipeline.fit(X_train, y_train, **fit_kwargs)
    else:
        pipeline.fit(X_train, y_train)
    joblib.dump(pipeline, checkpoint_path)
    return pipeline


def _extract_family(name: str) -> str:
    """'random_forest_balanced' -> 'random_forest'. Same-family runs
    (default/balanced) cost nearly identical time; different families can
    differ by 10x+ (e.g. Random Forest vs. GPU-accelerated CatBoost)."""
    return name.removesuffix("_default").removesuffix("_balanced")


def _eta_seconds(
    remaining_names: list[str],
    family_durations: dict[str, list[float]],
    last_duration: float | None,
) -> float | None:
    """Family-aware ETA.

    For each remaining stage, use its own family's observed average if
    we've already timed that family (default/balanced pairs cost almost
    the same). For a family not yet seen at all, fall back to the most
    recently observed single duration (of ANY family) rather than the
    overall historical average — so the estimate reacts within one step
    when training shifts from a slow family (Random Forest) to a fast one
    (XGBoost/LightGBM/CatBoost), instead of staying dragged down by the
    slow models seen earlier.
    """
    if last_duration is None:
        return None
    total = 0.0
    for name in remaining_names:
        family = _extract_family(name)
        family_seen = family_durations.get(family)
        total += (sum(family_seen) / len(family_seen)) if family_seen else last_duration
    return total


def run_stage(
    name: str,
    build_fn,
    fit_kwargs: dict | None,
    index: int,
    total: int,
    all_names: list[str],
    timing_state: dict,
):
    """Fit-or-load one model with family-aware progress/ETA logging, then score it.

    `timing_state` is a per-stage-group dict `{"family_durations": {}, "last_duration": None}`,
    shared and mutated across every call within one Stufe 0 / Stufe 1 loop
    (kept separate between the two loops since baselines and tree ensembles
    have unrelated cost profiles).
    """
    remaining_names = all_names[index:]  # stages after this one
    eta_seconds = _eta_seconds(
        remaining_names, timing_state["family_durations"], timing_state["last_duration"]
    )
    pct = 100 * (index - 1) / total
    eta_str = f"{eta_seconds / 60:.1f} min" if eta_seconds is not None else "unknown"
    _log_progress(f"[{index}/{total}] ({pct:.0f}%) training {name} ... ETA remaining: {eta_str}")
    start = time.time()
    pipeline = _fit_or_checkpoint(name, build_fn, fit_kwargs)
    elapsed = time.time() - start
    family = _extract_family(name)
    timing_state["family_durations"].setdefault(family, []).append(elapsed)
    timing_state["last_duration"] = elapsed
    _score_on_validation(name, pipeline)
    _log_progress(f"  -> {name} done in {elapsed:.1f}s")
    return pipeline

## 1 — Load the U-phase cache and apply the chronological split

A³ does not rebuild the DWD-enriched cache — it reads exactly what
`notebooks/02_U_Phase.ipynb` §8.5 already produced.

In [ ]:
_log_progress("Loading U-phase weather-enriched cache and applying chronological split...")
df = load_training_frame(BASE_DIR)
train_df, val_df, test_df = chronological_split(df)

X_train, y_train = split_features_target(train_df)
X_val, y_val = split_features_target(val_df)
X_test, y_test = split_features_target(test_df)

provenance = {
    "rows_train": len(X_train),
    "rows_val": len(X_val),
    "rows_test": len(X_test),
    "git_commit": _git_short_sha(),
    "run_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "random_seed": 42,
}
for k, v in provenance.items():
    print(f"  {k:14s} {v}")
_log_progress(
    f"Data loaded: {provenance['rows_train']:,} train / {provenance['rows_val']:,} val / {provenance['rows_test']:,} test rows."
)

## 2 — Cross-validation strategy

U-phase §10: "either a chronological TimeSeriesSplit or a year-grouped
K-fold; not a random StratifiedKFold." The training window spans 7
distinct years (2016–2022) — `GroupKFold` grouped by year gives 7 folds
that never let a model see a "future" year during model selection.

In [ ]:
cv_groups = train_df["UJAHR"].to_numpy()
cv = GroupKFold(n_splits=train_df["UJAHR"].nunique())
print(f"GroupKFold with {cv.get_n_splits(groups=cv_groups)} year-groups (2016-2022).")

## 3 — Stufe 0: baselines

Random guess and majority class establish the macro-F1 floor the Q-phase
expects (~0.33 and ~0.30 respectively); Logistic Regression is the first
non-trivial benchmark.

In [ ]:
tree_preprocessor = build_preprocessor(scale_for_linear=False)
linear_preprocessor = build_preprocessor(scale_for_linear=True)

comparison_rows: list[dict] = []


def _score_on_validation(name: str, fitted_estimator, X_val_override=None) -> None:
    X_to_use = X_val if X_val_override is None else X_val_override
    preds = fitted_estimator.predict(X_to_use)
    metrics = evaluate_predictions(y_val, preds)
    comparison_rows.append({"model": name, **metrics})
    print(
        f"{name:35s} macro-F1={metrics['macro_f1']:.3f}  recall(1)={metrics['recall_class_1']:.3f}"
    )


stufe0_specs = [
    ("random_guess", lambda: build_random_guess_classifier(), None),
    ("majority_class", lambda: build_majority_class_classifier(), None),
    ("logistic_regression", lambda: build_logreg_pipeline(linear_preprocessor), None),
]
stufe0_names = [name for name, _, _ in stufe0_specs]

fitted_baselines: dict = {}
_stufe0_timing: dict = {"family_durations": {}, "last_duration": None}
_log_progress(f"Starting Stufe 0: {len(stufe0_specs)} baselines.")
for i, (name, build_fn, fit_kwargs) in enumerate(stufe0_specs, start=1):
    fitted_baselines[name] = run_stage(
        name,
        build_fn,
        fit_kwargs,
        index=i,
        total=len(stufe0_specs),
        all_names=stufe0_names,
        timing_state=_stufe0_timing,
    )
_log_progress(
    f"Stufe 0 complete: {len(stufe0_specs)}/{len(stufe0_specs)} (100%) baselines trained."
)

random_guess = fitted_baselines["random_guess"]
majority_class = fitted_baselines["majority_class"]
logreg = fitted_baselines["logistic_regression"]

## 4 — Stufe 1: tree ensembles, default and class-weighted

Each of Random Forest, XGBoost, LightGBM, and CatBoost is trained twice:
once with library defaults, once with class-weighting applied — 8
configurations, all scored once against the 2023 validation split.

In [ ]:
train_class_counts = y_train.value_counts()
catboost_weights = [len(y_train) / (3 * train_class_counts[c]) for c in [1, 2, 3]]
xgb_weights = balanced_sample_weight(y_train)

stufe1_specs = [
    (
        "random_forest_default",
        lambda: build_random_forest_pipeline(tree_preprocessor, class_weight=None),
        None,
    ),
    (
        "random_forest_balanced",
        lambda: build_random_forest_pipeline(tree_preprocessor, class_weight="balanced"),
        None,
    ),
    (
        "xgboost_default",
        lambda: build_xgboost_pipeline(tree_preprocessor, use_gpu=_use_gpu_resolved),
        None,
    ),
    (
        "xgboost_balanced",
        lambda: build_xgboost_pipeline(tree_preprocessor, use_gpu=_use_gpu_resolved),
        {"classify__sample_weight": xgb_weights},
    ),
    (
        "lightgbm_default",
        lambda: build_lightgbm_pipeline(
            tree_preprocessor, class_weight=None, use_gpu=_use_gpu_resolved
        ),
        None,
    ),
    (
        "lightgbm_balanced",
        lambda: build_lightgbm_pipeline(
            tree_preprocessor, class_weight="balanced", use_gpu=_use_gpu_resolved
        ),
        None,
    ),
    (
        "catboost_default",
        lambda: build_catboost_pipeline(tree_preprocessor, use_gpu=_use_gpu_resolved),
        None,
    ),
    (
        "catboost_balanced",
        lambda: build_catboost_pipeline(
            tree_preprocessor, class_weights=catboost_weights, use_gpu=_use_gpu_resolved
        ),
        None,
    ),
]
stufe1_names = [name for name, _, _ in stufe1_specs]

fitted_models: dict = {}
_stufe1_timing: dict = {"family_durations": {}, "last_duration": None}
_log_progress(f"Starting Stufe 1: {len(stufe1_specs)} tree-ensemble configurations.")
for i, (name, build_fn, fit_kwargs) in enumerate(stufe1_specs, start=1):
    fitted_models[name] = run_stage(
        name,
        build_fn,
        fit_kwargs,
        index=i,
        total=len(stufe1_specs),
        all_names=stufe1_names,
        timing_state=_stufe1_timing,
    )
_log_progress(f"Stufe 1 complete: {len(stufe1_specs)}/{len(stufe1_specs)} (100%) models trained.")

## 5 — Champion selection

The champion is the single highest validation macro-F1 among the 8 tree
configurations above (baselines are never candidates for champion — they
exist to bound the floor, not to compete for it).

In [ ]:
comparison_df = pd.DataFrame(comparison_rows)
tree_only = comparison_df[comparison_df["model"].isin(fitted_models.keys())]
champion_name = tree_only.sort_values("macro_f1", ascending=False).iloc[0]["model"]
champion_pipeline = fitted_models[champion_name]
print(f"Champion (highest validation macro-F1 among tree configurations): {champion_name}")
comparison_df.sort_values("macro_f1", ascending=False)

In [ ]:
fig = px.bar(
    comparison_df.sort_values("macro_f1"),
    x="macro_f1",
    y="model",
    orientation="h",
    title="Validation macro-F1 by model (Stufe 0 + Stufe 1)",
)
fig.add_vline(x=0.55, line_dash="dash", annotation_text="Q-phase threshold (0.55)")
fig.write_html(FIG_DIR / "05_model_comparison_stufe0_1.html", include_plotlyjs="cdn")
fig.show()

> **Transition.** The champion architecture is selected on validation
> macro-F1. Part 2 of this notebook compares imbalance strategies on the
> champion only, tunes the winner, and reports the single test-2024
> evaluation.

## 6 — Imbalance-strategy comparison (champion only)

U-phase §10 menu, compared only on the champion's base estimator, on a
stratified subsample capped at 500,000 training rows (compute-budget soft
constraint, Q-phase §9). Class weights are already reflected by whichever
configuration won in §5 — this section adds SMOTE, ADASYN, threshold
moving, and ordinal classification on top of the *unweighted* variant of
the same model family, so all five configurations are comparable on equal
footing.

Reuses the same checkpoint-by-git-commit pattern from §0 (`_load_or_fit`
below) so a crash partway through this section does not require redoing
already-fitted strategies.

In [ ]:
def _load_or_fit(name: str, fit_callable):
    """Generic checkpoint helper for Part 2 (arbitrary X/y per stage, unlike
    `_fit_or_checkpoint` in §0 which is hardcoded to X_train/y_train)."""
    path = CHECKPOINT_DIR / f"{name}.joblib"
    if path.exists():
        _log_progress(f"  -> {name}: loaded from checkpoint ({path.name})")
        return joblib.load(path)
    start = time.time()
    model = fit_callable()
    elapsed = time.time() - start
    joblib.dump(model, path)
    _log_progress(f"  -> {name} done in {elapsed:.1f}s")
    return model


SUBSAMPLE_CAP = 500_000
if len(X_train) > SUBSAMPLE_CAP:
    sample_idx = (
        y_train.groupby(y_train)
        .apply(lambda s: s.sample(frac=min(1.0, SUBSAMPLE_CAP / len(y_train)), random_state=42))
        .index.get_level_values(-1)
    )
    X_train_sub = X_train.loc[sample_idx].reset_index(drop=True)
    y_train_sub = y_train.loc[sample_idx].reset_index(drop=True)
    years_sub = train_df.loc[sample_idx, "UJAHR"].reset_index(drop=True)
else:
    X_train_sub, y_train_sub = X_train.reset_index(drop=True), y_train.reset_index(drop=True)
    years_sub = train_df["UJAHR"].reset_index(drop=True)
_log_progress(f"Imbalance-comparison subsample: {len(X_train_sub):,} rows (cap={SUBSAMPLE_CAP:,}).")


def _log_section6_progress(
    step_idx: int, total_steps: int, name: str, timing_state: dict, all_names: list[str]
) -> None:
    remaining = all_names[step_idx:]
    eta = _eta_seconds(remaining, timing_state["family_durations"], timing_state["last_duration"])
    eta_str = f"{eta / 60:.1f} min" if eta is not None else "unknown"
    pct = 100 * step_idx / total_steps
    _log_progress(
        f"[{step_idx + 1}/{total_steps}] ({pct:.0f}%) {name} ... ETA remaining: {eta_str}"
    )


def _log_section6_done(name: str, elapsed: float, timing_state: dict) -> None:
    timing_state["family_durations"].setdefault(name, []).append(elapsed)
    timing_state["last_duration"] = elapsed
    _log_progress(f"  -> {name} done in {elapsed:.1f}s")

In [ ]:
# NOTE: build_champion(tree_preprocessor) must return the *unweighted*
# variant for a fair comparison against SMOTE/ADASYN/ordinal (which are
# themselves the imbalance treatment). Only CatBoost's own default
# (class_weights=None) and XGBoost's builder (no class_weight concept at
# all) are unweighted by default - Random Forest and LightGBM both default
# to class_weight="balanced" in their builders, so that must be overridden
# explicitly here or this comparison would silently double up on class
# weighting for those two families.
champion_builder = {
    "random_forest": lambda pre, **kw: build_random_forest_pipeline(pre, class_weight=None, **kw),
    "xgboost": lambda pre, **kw: build_xgboost_pipeline(pre, use_gpu=_use_gpu_resolved, **kw),
    "lightgbm": lambda pre, **kw: build_lightgbm_pipeline(
        pre, class_weight=None, use_gpu=_use_gpu_resolved, **kw
    ),
    "catboost": lambda pre, **kw: build_catboost_pipeline(pre, use_gpu=_use_gpu_resolved, **kw),
}
champion_family = _extract_family(champion_name)
build_champion = champion_builder[champion_family]
print(f"Champion family: {champion_family}")

_section6_names = [
    f"{champion_family}_smote",
    f"{champion_family}_adasyn",
    f"{champion_family}_threshold_moving",
    f"{champion_family}_ordinal",
]
_section6_timing: dict = {"family_durations": {}, "last_duration": None}

# SMOTE/ADASYN's k-NN search requires finite numeric input. IstGkfz is
# genuinely NaN for ~12.6% of rows (only recorded from 2018 onward, per
# docs/GLOSSARY.md) and tree_preprocessor's passthrough branch
# (scale_for_linear=False) deliberately leaves it untouched so RF/XGBoost/
# LightGBM/CatBoost can use it as a native split signal - that's why Stufe
# 0/1 never crashed on it. SMOTE/ADASYN have no such native NaN handling
# and raise "ValueError: Input X contains NaN" if fed this output directly.
#
# A plain SimpleImputer is NOT sufficient here: ColumnTransformer's hstack
# of the passthrough bool/NaN column with the float columns produces an
# object-dtype array, and confirmed via direct inspection, IstGkfz's
# missing entries in that object array are Python `None`, not float
# `np.nan`. SimpleImputer's default NaN detection relies on
# self-inequality (`x != x`), which is only true for float NaN - `None`
# silently survives untouched, leaving residual NaN that crashes SMOTE/
# ADASYN downstream even after "imputing". Coercing through
# `pd.to_numeric(errors="coerce")` first canonicalizes every missing
# representation (None, np.nan, pd.NA, ...) to a proper np.nan, which
# SimpleImputer then correctly detects and fills.
fitted_preprocessor_for_resampling = clone(tree_preprocessor).fit(X_train_sub, y_train_sub)
_resampling_imputer = SimpleImputer(strategy="most_frequent")


def _numeric_impute(raw_transformed: np.ndarray, fit: bool) -> np.ndarray:
    numeric = pd.DataFrame(raw_transformed).apply(pd.to_numeric, errors="coerce")
    if fit:
        return _resampling_imputer.fit_transform(numeric)
    return _resampling_imputer.transform(numeric)


X_train_sub_transformed = _numeric_impute(
    fitted_preprocessor_for_resampling.transform(X_train_sub), fit=True
)
X_val_transformed_for_resampling = _numeric_impute(
    fitted_preprocessor_for_resampling.transform(X_val), fit=False
)


def build_champion_classifier_only():
    """Unfitted classify-step estimator only, for direct use on already-
    preprocessed numeric arrays (SMOTE/ADASYN output) - a full
    build_champion(tree_preprocessor) Pipeline would try to re-preprocess
    the already-transformed array as if it were the raw DataFrame."""
    return build_champion(tree_preprocessor).named_steps["classify"]

In [ ]:
# 6a: SMOTE
_log_section6_progress(0, 4, f"{champion_family}_smote", _section6_timing, _section6_names)
_step_start = time.time()
X_smote, y_smote = resample_smote(X_train_sub_transformed, y_train_sub)
model_smote = _load_or_fit(
    f"{champion_family}_smote", lambda: build_champion_classifier_only().fit(X_smote, y_smote)
)
_score_on_validation(
    f"{champion_family}_smote", model_smote, X_val_override=X_val_transformed_for_resampling
)
_log_section6_done(f"{champion_family}_smote", time.time() - _step_start, _section6_timing)

In [ ]:
# 6b: ADASYN
_log_section6_progress(1, 4, f"{champion_family}_adasyn", _section6_timing, _section6_names)
_step_start = time.time()
X_adasyn, y_adasyn = resample_adasyn(X_train_sub_transformed, y_train_sub)
model_adasyn = _load_or_fit(
    f"{champion_family}_adasyn", lambda: build_champion_classifier_only().fit(X_adasyn, y_adasyn)
)
_score_on_validation(
    f"{champion_family}_adasyn", model_adasyn, X_val_override=X_val_transformed_for_resampling
)
_log_section6_done(f"{champion_family}_adasyn", time.time() - _step_start, _section6_timing)

In [ ]:
# 6c: threshold moving (post-hoc on the unweighted champion-family model)
_log_section6_progress(
    2, 4, f"{champion_family}_threshold_moving", _section6_timing, _section6_names
)
_step_start = time.time()
model_unweighted = _load_or_fit(
    f"{champion_family}_unweighted",
    lambda: build_champion(tree_preprocessor).fit(X_train_sub, y_train_sub),
)
val_proba = model_unweighted.predict_proba(X_val)
best_threshold = find_best_threshold_for_class(
    y_val, val_proba, classes=list(model_unweighted.classes_), target_class=1
)
target_idx = list(model_unweighted.classes_).index(1)
other_classes = np.array([c for c in model_unweighted.classes_ if c != 1])
other_proba = np.delete(val_proba, target_idx, axis=1)
fallback = other_classes[np.argmax(other_proba, axis=1)]
threshold_preds = np.where(val_proba[:, target_idx] >= best_threshold, 1, fallback)
threshold_metrics = evaluate_predictions(y_val, threshold_preds)
comparison_rows.append({"model": f"{champion_family}_threshold_moving", **threshold_metrics})
print(f"Best threshold for class 1: {best_threshold:.2f}")
_log_section6_done(
    f"{champion_family}_threshold_moving", time.time() - _step_start, _section6_timing
)

In [ ]:
# 6d: ordinal classification (Frank-Hall) using the champion family's own estimator as base
_log_section6_progress(3, 4, f"{champion_family}_ordinal", _section6_timing, _section6_names)
_step_start = time.time()
base_estimator_for_ordinal = clone(build_champion(tree_preprocessor).named_steps["classify"])
ordinal_pipeline = _load_or_fit(
    f"{champion_family}_ordinal",
    lambda: build_ordinal_pipeline(tree_preprocessor, base_estimator_for_ordinal).fit(
        X_train_sub, y_train_sub
    ),
)
_score_on_validation(f"{champion_family}_ordinal", ordinal_pipeline)
_log_section6_done(f"{champion_family}_ordinal", time.time() - _step_start, _section6_timing)

comparison_df = pd.DataFrame(comparison_rows)
strategy_rows = comparison_df[comparison_df["model"].str.startswith(champion_family)]
winning_strategy_row = strategy_rows.sort_values("macro_f1", ascending=False).iloc[0]
_log_progress(f"Winning (model, strategy) combination: {winning_strategy_row['model']}")
print(f"Winning (model, strategy) combination: {winning_strategy_row['model']}")
strategy_rows.sort_values("macro_f1", ascending=False)


def _build_winning_pipeline(pre):
    """Unfitted pipeline matching whichever (model, strategy) combination
    actually WON the §6 comparison above - not `build_champion`'s
    always-unweighted baseline, which exists only to give SMOTE/ADASYN/
    threshold-moving/ordinal a fair, equally-unweighted opponent in that
    comparison. Tuning/refitting `build_champion` directly here would
    silently retune the wrong (unweighted) configuration regardless of
    which strategy actually won - e.g. if plain class_weight="balanced"
    (`champion_pipeline`) beat every resampling/ordinal treatment, §7/§8
    must tune and refit *that*, not the unweighted variant.
    """
    winner = winning_strategy_row["model"]
    if winner == champion_name:
        return clone(champion_pipeline)
    if winner == f"{champion_family}_unweighted":
        return build_champion(pre)
    raise NotImplementedError(
        f"No tuning/refit path implemented for winning strategy '{winner}' - "
        "SMOTE/ADASYN/threshold-moving/ordinal each need a fitting procedure "
        "inside Optuna's per-fold CV other than plain "
        "Pipeline.set_params().fit(), which is out of scope here."
    )

## 7 — Hyperparameter tuning (Optuna, winning combination only)

Bounded to 40 trials on the same subsample, optimising mean validation
macro-F1 across the year-grouped folds from §2. Only the single winning
(model, strategy) combination from §6 is tuned — not every configuration
trained so far (Q-phase §9 compute-budget constraint).

**GPU reminder.** `objective()` below calls `build_champion(tree_preprocessor)`
once per trial per fold (`n_trials x n_groups_sub` fits total) — by far the
most fit-heavy loop in this notebook. `champion_builder` (§6) already
threads `_use_gpu_resolved` into every XGBoost/LightGBM/CatBoost call, so
this loop runs on GPU automatically whenever `USE_GPU` resolves to `True`.

**Parameter-routing note.** XGBoost's classify step is wrapped by
`_ZeroIndexedXGBClassifier` (label-safety fix, see `src/unfallatlas/models/boosting.py`),
which only exposes `estimator` as its own constructor parameter — so tuning
its nested `XGBClassifier` requires `classify__estimator__<param>`, not
`classify__<param>` (verified empirically; the latter raises
`ValueError: Invalid parameter`). Random Forest/LightGBM/CatBoost have no
such wrapper, so `classify__<param>` is correct for those three.

The study is persisted to a SQLite file under the same commit-scoped
`CHECKPOINT_DIR` used throughout this notebook, so a crash mid-tuning
resumes from the last completed trial instead of restarting all 40.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 18

PARAM_SPACES = {
    "random_forest": lambda trial: {
        "classify__n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "classify__max_depth": trial.suggest_int("max_depth", 4, 20),
        "classify__min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "classify__max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None, 0.3, 0.5]
        ),
    },
    "xgboost": lambda trial: {
        "classify__estimator__n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "classify__estimator__max_depth": trial.suggest_int("max_depth", 3, 10),
        "classify__estimator__learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),
    },
    "lightgbm": lambda trial: {
        "classify__n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "classify__num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "classify__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
    },
    "catboost": lambda trial: {
        "classify__iterations": trial.suggest_int("iterations", 100, 500),
        "classify__depth": trial.suggest_int("depth", 3, 10),
        "classify__learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
    },
}


n_groups_sub = min(5, years_sub.nunique())
print(
    f"Optuna CV folds (n_groups_sub): {n_groups_sub}; distinct years in subsample: {years_sub.nunique()}"
)


def objective(trial: optuna.Trial) -> float:
    params = PARAM_SPACES[champion_family](trial)
    pipeline = _build_winning_pipeline(tree_preprocessor).set_params(**params)
    scores = cross_val_score(
        pipeline,
        X_train_sub,
        y_train_sub,
        cv=GroupKFold(n_splits=n_groups_sub),
        groups=years_sub,
        scoring="f1_macro",
        n_jobs=1,
    )
    return float(scores.mean())


_optuna_trial_durations: list[float] = []


def _optuna_progress_callback(study: "optuna.Study", trial: "optuna.trial.FrozenTrial") -> None:
    elapsed = trial.duration.total_seconds() if trial.duration else 0.0
    _optuna_trial_durations.append(elapsed)
    avg = sum(_optuna_trial_durations) / len(_optuna_trial_durations)
    remaining = N_TRIALS - (trial.number + 1)
    eta_min = (avg * remaining) / 60
    _log_progress(
        f"[Optuna {trial.number + 1}/{N_TRIALS}] trial macro-F1={trial.value:.3f} "
        f"(best so far={study.best_value:.3f}) in {elapsed:.1f}s ... ETA remaining: {eta_min:.1f} min"
    )


optuna_db_path = CHECKPOINT_DIR / "optuna_study.db"
study = optuna.create_study(
    study_name="a3_champion_tuning",
    storage=f"sqlite:///{optuna_db_path}",
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
remaining_trials = max(0, N_TRIALS - len(study.trials))
_log_progress(
    f"Optuna study: {len(study.trials)}/{N_TRIALS} trials already completed (persisted at {optuna_db_path.name}); {remaining_trials} remaining."
)
if remaining_trials > 0:
    study.optimize(objective, n_trials=remaining_trials, callbacks=[_optuna_progress_callback])
print(f"Best trial macro-F1 (CV, subsample): {study.best_value:.3f}")
print(f"Best params: {study.best_params}")

## 8 — Refit on full training data and evaluate on test-2024 exactly once

The tuned configuration is refit on the **full** 2016–2022 training set
(not the subsample used for tuning), then evaluated on the 2024 test
split — the single time this notebook touches the test set.

In [ ]:
_log_progress("Refitting tuned configuration on the FULL 2016-2022 training set...")
best_params = {f"classify__{k}": v for k, v in study.best_params.items()}
if champion_family == "xgboost":
    best_params = {
        k.replace("classify__", "classify__estimator__", 1): v for k, v in best_params.items()
    }

final_pipeline = _load_or_fit(
    f"{champion_family}_final_tuned",
    lambda: (
        _build_winning_pipeline(tree_preprocessor).set_params(**best_params).fit(X_train, y_train)
    ),
)

test_preds = final_pipeline.predict(X_test)
final_metrics = evaluate_predictions(y_test, test_preds)
passes = meets_acceptance_criteria(final_metrics)

_log_progress(
    f"FINAL TEST-2024: macro-F1={final_metrics['macro_f1']:.3f} recall(1)={final_metrics['recall_class_1']:.3f} "
    f"gate={'PASS' if passes else 'FAIL'}"
)
print("=== FINAL TEST-2024 EVALUATION ===")
print(f"macro-F1:        {final_metrics['macro_f1']:.3f}  (threshold >= 0.55)")
print(f"recall(class 1): {final_metrics['recall_class_1']:.3f}  (threshold >= 0.50)")
print(f"Q-phase acceptance gate: {'PASS' if passes else 'FAIL'}")
print("Confusion matrix (rows=true, cols=pred, order=[1,2,3]):")
print(np.array(final_metrics["confusion_matrix"]))

## 9 — Save the winning pipeline and model card

In [ ]:
model_path = PROCESSED_DIR / "a3_best_model.joblib"
joblib.dump(final_pipeline, model_path)

model_card = {
    "champion_family": champion_family,
    "winning_strategy": winning_strategy_row["model"],
    "tuned_hyperparameters": study.best_params,
    "test_2024_metrics": final_metrics,
    "acceptance_gate_passed": passes,
    "provenance": provenance,
}
card_path = PROCESSED_DIR / "a3_model_card.json"
card_path.write_text(json.dumps(model_card, indent=2))
model_size_mb = model_path.stat().st_size / 1_048_576
_log_progress(f"Saved {model_path.name} ({model_size_mb:.1f} MB) and {card_path.name}.")
print(f"Saved: {model_path} ({model_size_mb:.1f} MB)")
print(f"Saved: {card_path}")

## 10 — A³ summary and A³-to-C handoff

### Model-comparison table (central portfolio artefact)

See §5 and §6 above — 8 baseline/tree configurations + 4 imbalance
strategies on the champion + 1 tuned final configuration, all scored on
the 2023 validation split, with exactly one test-2024 evaluation.

### A³-phase acceptance checklist

```text
[ ] U-phase §10 preprocessing implemented as a single sklearn Pipeline
[ ] Baselines (random guess, majority class, logistic regression) scored
[ ] 4 tree families x {default, class-weighted} scored on validation
[ ] Champion selected by validation macro-F1
[ ] SMOTE / ADASYN / threshold moving / ordinal classification compared
    on the champion only
[ ] Winning (model, strategy) combination tuned with Optuna (<= 40 trials)
[ ] Tuned configuration refit on the FULL 2016-2022 training set
[ ] Exactly one evaluation on the 2024 test set
[ ] PASS/FAIL stated explicitly against macro-F1 >= 0.55 and recall(1) >= 0.50
[ ] Winning pipeline + model card saved to data/processed/
[ ] No SHAP, no literature discussion, no Streamlit work in this notebook
```

> **Transition.** The winning pipeline and its model card are saved to
> `data/processed/`. Proceed to `04_C_Phase.ipynb` for SHAP-based
> explainability, benchmark comparison against the literature anchor
> (Q-phase §11), and the limitations discussion.